# Baseline CNN Model

A simple Convolutional Neural Network to establish a baseline performance on the emotion recognition task.

**Objectives:**
- Build a simple 3-layer CNN architecture
- Train the model with data augmentation
- Use early stopping and learning rate reduction
- Evaluate baseline performance
- Save the trained model

In [1]:
import os
import sys
import json
import numpy as np
import tensorflow as tf

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src import model as model_module, train as train_module

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU Available: []


## Load Preprocessed Data

In [2]:
# Load preprocessed data
data_dir = os.path.join('..', 'data', 'preprocessed')

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

# Load class weights
with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

print(f"Training data: {X_train.shape}")
print(f"Validation data: {X_val.shape}")
print(f"Test data: {X_test.shape}")
print(f"\nClass weights loaded: {len(class_weights)} classes")

Training data: (22968, 48, 48, 1)
Validation data: (5741, 48, 48, 1)
Test data: (7178, 48, 48, 1)

Class weights loaded: 7 classes


## Build Baseline Model

In [3]:
# Build baseline CNN
baseline_model = model_module.baseline_cnn(input_shape=(48, 48, 1), num_classes=7)
baseline_model.summary()

C:\Users\Yasmine Sassi\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 619,911 (2.36 MB)

 Trainable params: 619,463 (2.36 MB)

 Non-trainable params: 448 (1.75 KB)

## Train Model

In [4]:
# Train baseline model
print("Training baseline model...")
history_baseline = train_module.train_model(
    baseline_model,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    class_weights=class_weights,
    model_name='baseline_cnn'
)

Training baseline model...
Epoch 1/100
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.2494 - loss: 2.0389
Epoch 1: val_accuracy improved from None to 0.25030, saving model to saved_models/baseline_cnn_best.keras

Epoch 1: finished saving model to saved_models/baseline_cnn_best.keras
718/718 ━━━━━━━━━━━━━━━━━━━━ 76s 100ms/step - accuracy: 0.2766 - loss: 1.8270 - val_accuracy: 0.2503 - val_loss: 1.7653 - learning_rate: 0.0010
Epoch 2/100
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.3296 - loss: 1.6737
Epoch 2: val_accuracy improved from 0.25030 to 0.32695, saving model to saved_models/baseline_cnn_best.keras

Epoch 2: finished saving model to saved_models/baseline_cnn_best.keras
718/718 ━━━━━━━━━━━━━━━━━━━━ 67s 79ms/step - accuracy: 0.3456 - loss: 1.6461 - val_accuracy: 0.3269 - val_loss: 1.6441 - learning_rate: 0.0010
Epoch 3/100
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3786 - loss: 1.5815
Epoch 3: val_accuracy improved from 0.32695 to 0.40690, saving 

## Visualize Training Results

In [7]:
# Plot training history
train_module.plot_training_history(history_baseline, save_path='../results/model/baseline_training_history.png')

Plot saved to ../results/model/baseline_training_history.png


## Evaluate on Test Set

In [8]:
from src.evaluate import evaluate_model, plot_confusion_matrix

# Evaluate baseline model
print("Evaluating baseline model on test set...")
results_baseline = evaluate_model(baseline_model, X_test, y_test, model_name='baseline')

# Plot confusion matrix
plot_confusion_matrix(y_test, results_baseline['y_pred'], model_name='baseline')

Evaluating baseline model on test set...
225/225 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step

Test Accuracy: 0.4947

Classification Report:
              precision    recall  f1-score   support

       Angry       0.39      0.28      0.32       958
     Disgust       0.00      0.00      0.00       111
        Fear       0.26      0.19      0.22      1024
       Happy       0.70      0.76      0.73      1774
     Neutral       0.42      0.65      0.51      1233
         Sad       0.42      0.23      0.30      1247
    Surprise       0.52      0.78      0.62       831

    accuracy                           0.49      7178
   macro avg       0.39      0.41      0.39      7178
weighted avg       0.47      0.49      0.47      7178



C:\Users\Yasmine Sassi\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Yasmine Sassi\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Yasmine Sassi\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

Confusion matrix saved to results/baseline_confusion_matrix.png
